# Evaluation Diagnostics Analysis

This notebook loads `evaluation_mismatch_diagnostics.csv` files produced by `evaluation.py --diagnostics`, quantifies mismatch categories, and provides visual drilldowns for manual review.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 220)

ROOT = Path.cwd()
if not (ROOT / "evaluation.py").exists():
    ROOT = ROOT.parent.parent

RESULTS_DIR = ROOT / "results"
RESULTS_DIR

## Load Diagnostics

Run evaluation first, for example:

```powershell
python evaluation.py -d .\results\nl2p_1\gpt-5-mini --diagnostics
```

In [ ]:
def load_diagnostics(results_dir=RESULTS_DIR):
    paths = sorted(Path(results_dir).rglob("evaluation_mismatch_diagnostics.csv"))
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["diagnostics_file"] = str(path.relative_to(ROOT))
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f"No evaluation_mismatch_diagnostics.csv found under {results_dir}")
    return pd.concat(frames, ignore_index=True)

diag = load_diagnostics()
diag.head()

In [ ]:
diag.info()
diag[["dataset", "solver", "model", "mismatch_type", "candidate_dataset_issue", "candidate_llm_issue"]].describe(include="all")

## Normalize Multi-Issue Columns

In [ ]:
def split_issue_column(df, column):
    out = df[["dataset", "solver", "model", "doc_id", "docId", "mismatch_type", column, "diagnostics_file"]].copy()
    out[column] = out[column].fillna("").astype(str)
    out[column] = out[column].str.split("|")
    out = out.explode(column)
    out[column] = out[column].str.strip()
    out = out[out[column] != ""]
    return out

dataset_issues = split_issue_column(diag, "candidate_dataset_issue")
llm_issues = split_issue_column(diag, "candidate_llm_issue")

display(dataset_issues.head())
display(llm_issues.head())

## Summary Tables

In [ ]:
summary = {
    "num_rows": len(diag),
    "num_docs": diag[["dataset", "doc_id"]].drop_duplicates().shape[0],
    "datasets": sorted(diag["dataset"].dropna().unique().tolist()),
    "models": sorted(diag["model"].dropna().unique().tolist()),
}
summary

In [ ]:
mismatch_counts = (
    diag.groupby(["dataset", "solver", "model", "mismatch_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
mismatch_counts

In [ ]:
dataset_issue_counts = (
    dataset_issues.groupby(["dataset", "solver", "model", "candidate_dataset_issue"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
llm_issue_counts = (
    llm_issues.groupby(["dataset", "solver", "model", "candidate_llm_issue"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(dataset_issue_counts.head(30))
display(llm_issue_counts.head(30))

## Visualize Mismatch Types

In [ ]:
plt.figure(figsize=(10, 5))
order = diag["mismatch_type"].value_counts().index
sns.countplot(data=diag, y="mismatch_type", order=order, hue="dataset")
plt.title("Mismatch Types by Dataset")
plt.xlabel("Count")
plt.ylabel("Mismatch Type")
plt.legend(title="Dataset")
plt.tight_layout()

In [ ]:
def plot_issue_counts(issue_df, issue_col, title, top_n=20):
    if issue_df.empty:
        print(f"No rows for {issue_col}")
        return
    counts = issue_df[issue_col].value_counts().head(top_n).reset_index()
    counts.columns = [issue_col, "count"]
    plt.figure(figsize=(11, max(4, 0.35 * len(counts))))
    sns.barplot(data=counts, x="count", y=issue_col, color="#4C78A8")
    plt.title(title)
    plt.xlabel("Count")
    plt.ylabel("")
    plt.tight_layout()

plot_issue_counts(dataset_issues, "candidate_dataset_issue", "Candidate Dataset Annotation Issues")
plot_issue_counts(llm_issues, "candidate_llm_issue", "Candidate LLM Extraction Issues")

## Dataset vs LLM Issue Balance

In [ ]:
issue_balance = pd.concat(
    [
        dataset_issues.assign(source="dataset", issue=dataset_issues["candidate_dataset_issue"]),
        llm_issues.assign(source="llm", issue=llm_issues["candidate_llm_issue"]),
    ],
    ignore_index=True,
)

balance_counts = (
    issue_balance.groupby(["dataset", "source"])
    .size()
    .reset_index(name="count")
)

plt.figure(figsize=(8, 4))
sns.barplot(data=balance_counts, x="dataset", y="count", hue="source")
plt.title("Candidate Issue Counts: Dataset Annotation vs LLM Extraction")
plt.xlabel("Dataset")
plt.ylabel("Count")
plt.tight_layout()

balance_counts

## Documents With Most Diagnostics

In [ ]:
doc_counts = (
    diag.groupby(["dataset", "doc_id", "docId"])
    .size()
    .reset_index(name="diagnostic_count")
    .sort_values("diagnostic_count", ascending=False)
)
doc_counts.head(30)

In [ ]:
top_docs = doc_counts.head(20)
plt.figure(figsize=(11, 6))
sns.barplot(data=top_docs, x="diagnostic_count", y="docId", hue="dataset", dodge=False)
plt.title("Top Documents by Number of Diagnostics")
plt.xlabel("Diagnostics")
plt.ylabel("docId")
plt.tight_layout()

## Drilldown Helpers

In [ ]:
def show_doc(doc_id=None, docId=None, dataset=None, columns=None):
    columns = columns or [
        "dataset", "doc_id", "docId", "mismatch_type",
        "candidate_dataset_issue", "candidate_llm_issue", "reason",
        "gold_verb", "gold_arguments", "pred_verb", "pred_arguments",
        "original_text",
    ]
    mask = pd.Series(True, index=diag.index)
    if doc_id is not None:
        mask &= diag["doc_id"].astype(str) == str(doc_id)
    if docId is not None:
        mask &= diag["docId"].astype(str) == str(docId)
    if dataset is not None:
        mask &= diag["dataset"].astype(str) == str(dataset)
    return diag.loc[mask, columns]

def sample_rows(issue=None, mismatch_type=None, dataset=None, n=10, random_state=0):
    mask = pd.Series(True, index=diag.index)
    if issue is not None:
        issue_mask = diag["candidate_dataset_issue"].fillna("").str.contains(issue, regex=False)
        issue_mask |= diag["candidate_llm_issue"].fillna("").str.contains(issue, regex=False)
        mask &= issue_mask
    if mismatch_type is not None:
        mask &= diag["mismatch_type"] == mismatch_type
    if dataset is not None:
        mask &= diag["dataset"] == dataset
    rows = diag.loc[mask]
    if len(rows) <= n:
        return rows
    return rows.sample(n=n, random_state=random_state)

sample_rows(n=5)

Example drilldowns:

In [ ]:
# show_doc(docId="cooking:118")
# sample_rows(issue="missing_arguments", n=10)
# sample_rows(issue="extra_arguments:preposition_argument", n=10)
# sample_rows(mismatch_type="wrong_action", n=10)

## Export Review Samples

In [ ]:
REVIEW_DIR = ROOT / "results" / "diagnostics_review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

review_sample = pd.concat(
    [
        sample_rows(issue="missing_arguments", n=20, random_state=1),
        sample_rows(issue="extra_arguments", n=20, random_state=2),
        sample_rows(issue="wrong_actions", n=20, random_state=3),
    ],
    ignore_index=True,
).drop_duplicates()

review_path = REVIEW_DIR / "diagnostics_review_sample.csv"
review_sample.to_csv(review_path, index=False)
review_path